# Understanding IFC Files: Building Code Compliance Exercises

## What is IFC?

**IFC (Industry Foundation Classes)** is a standard data format for building information. It stores geometric, spatial, and semantic information about buildings in a structured way. Think of it as a database format for architectural and engineering data.

Industries use IFC to:
- Share building models across different software
- Extract specific information (spaces, materials, systems)
- Validate designs against building codes
- Run simulations and analysis

## Resources

Before starting, explore these resources to understand IFC better:

- **IFC Knowledge Base**: https://notebooklm.google.com/notebook/0925c2a1-519b-40a8-aca4-1e832d219f3c
- **BuildingSmart (IFC Standard)**: https://www.buildingsmart.org/standards/bsi-standards/industry-foundation-classes/
- **IfcOpenShell Python Docs**: https://docs.ifcopenshell.org/ifcopenshell-python.html
- **Catalan Building Code Reference**: https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e (for exercises 1 & bonus)

This notebook uses **IfcOpenShell**, a Python library for reading and writing IFC files.

## Setup: Load and Explore IFC Files

First, let's install IfcOpenShell and load the duplex apartment model.

In [1]:
# Install IfcOpenShell (run this once)
import subprocess
import sys

try:
    import ifcopenshell
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ifcopenshell", "-q"])
    import ifcopenshell

# Import other useful libraries
import json
from pathlib import Path
from collections import defaultdict

# Load the IFC file
ifc_file_path = Path("assets/duplex.ifc")
ifc = ifcopenshell.open(ifc_file_path)

print(f"✓ IFC file loaded: {ifc_file_path}")
print(f"✓ IFC Schema: {ifc.schema}")

✓ IFC file loaded: assets/duplex.ifc
✓ IFC Schema: IFC2X3


### What's Inside the File?

Let's explore the structure of the IFC model:

In [6]:
# Get all unique entity types in the model
all_entities = list(ifc)
entity_types = defaultdict(int)
for entity in all_entities:
    entity_types[entity.is_a()] += 1

# Show entity counts
print("Entity types in this model:")
print("-" * 40)
for entity_type in sorted(entity_types.keys()):
    count = entity_types[entity_type]
    print(f"  {entity_type}: {count}")

print(f"\nTotal entities: {len(all_entities)}")

# Find basic info
building = ifc.by_type("IfcBuilding")[0] if ifc.by_type("IfcBuilding") else None
if building:
    print(f"\nBuilding name: {building.Name}")
    print(f"Building description: {building.Description}")

Entity types in this model:
----------------------------------------
  IfcApplication: 1
  IfcArbitraryClosedProfileDef: 102
  IfcArbitraryOpenProfileDef: 184
  IfcArbitraryProfileDefWithVoids: 18
  IfcAxis2Placement2D: 397
  IfcAxis2Placement3D: 1279
  IfcBeam: 8
  IfcBooleanClippingResult: 7
  IfcBuilding: 1
  IfcBuildingStorey: 4
  IfcCartesianPoint: 8520
  IfcCartesianTransformationOperator3D: 167
  IfcCircle: 96
  IfcCircleProfileDef: 8
  IfcColourRgb: 43
  IfcCompositeCurve: 44
  IfcCompositeCurveSegment: 322
  IfcConnectedFaceSet: 235
  IfcConnectionSurfaceGeometry: 265
  IfcConversionBasedUnit: 1
  IfcCovering: 13
  IfcCurveBoundedPlane: 81
  IfcCurveStyle: 31
  IfcCurveStyleFont: 19
  IfcCurveStyleFontPattern: 57
  IfcDimensionalExponents: 1
  IfcDirection: 134
  IfcDoor: 14
  IfcDoorLiningProperties: 6
  IfcDoorStyle: 6
  IfcDraughtingPreDefinedCurveFont: 1
  IfcElementQuantity: 21
  IfcExtrudedAreaSolid: 421
  IfcFace: 4486
  IfcFaceBasedSurfaceModel: 40
  IfcFaceBound: 60
 

## Example: Extract and Display All IFC Spaces

An **IfcSpace** represents a room or area in the building. Each space has properties like name, area, and volume. Here's how to extract them:

In [3]:
# Get all spaces
spaces = ifc.by_type("IfcSpace")

print(f"Found {len(spaces)} spaces in the building\n")
print("=" * 60)

for i, space in enumerate(spaces, 1):
    # Extract properties
    name = space.Name if space.Name else "Unnamed"
    
    # Try to get area and volume
    area = None
    volume = None
    
    if hasattr(space, 'Quantities') and space.Quantities:
        for quantity in space.Quantities.Quantities:
            if hasattr(quantity, 'Name'):
                if quantity.Name == "NetFloorArea" and hasattr(quantity, 'AreaValue'):
                    area = quantity.AreaValue
                elif quantity.Name == "GrossVolume" and hasattr(quantity, 'VolumeValue'):
                    volume = quantity.VolumeValue
    
    # Format output
    print(f"{i}. {name}")
    if area:
        print(f"   Area: {area:.2f} m²")
    if volume:
        print(f"   Volume: {volume:.2f} m³")
    print()

print("=" * 60)

Found 21 spaces in the building

1. A101

2. A201

3. B104

4. B101

5. B201

6. A205

7. B205

8. A105

9. B105

10. R301

11. B102

12. A102

13. A103

14. B103

15. B204

16. A204

17. A104

18. B203

19. A202

20. B202

21. A203



## Exercise 1: Building Code Compliance Checker (Catalonia)

**Objective:** Write a function that validates spaces against Catalonia building code requirements.

**Reference:** [Catalan Building Code](https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e)

### Key Requirements to Check

Based on the Catalan building code, residential spaces should meet these minimum standards:

| Space Type | Min Height | Min Area | Purpose |
|---|---|---|---|
| Living Room | 2.6 m | 16 m² | Main living area |
| Bedroom | 2.6 m | 9 m² | Single occupancy |
| Kitchen | 2.6 m | 8 m² | Cooking/dining |
| Bathroom | 2.3 m | 4 m² | Hygiene |
| Corridor | 2.3 m | 1.5 m² | Circulation |

### Your Task

Write a function `check_space_compliance(spaces)` that:

1. **Identifies** each space type (you can infer from the name or classify them)
2. **Extracts** the height and area properties from each space
3. **Compares** against the requirements above
4. **Reports** which spaces pass/fail and why
5. **Returns** a summary with compliance status

**Hints:**
- Height can be extracted from the Z-coordinate range of the space geometry
- Area is usually stored in space.Quantities as "NetFloorArea"
- Space names often indicate their type (e.g., "Master Bedroom", "Kitchen")

### Starter Code

In [10]:
# Exercise 1: Implement your solution here
# Copy the starter code below and complete the function

def check_space_compliance(spaces):
    """
    Check each space for Catalan building code compliance.

    Args:
        spaces: List of IfcSpace objects

    Returns:
        dict: Compliance report
    """

    requirements = {
        "Living Room": {"min_height": 2.6, "min_area": 16},
        "Bedroom": {"min_height": 2.6, "min_area": 9},
        "Kitchen": {"min_height": 2.6, "min_area": 8},
        "Bathroom": {"min_height": 2.3, "min_area": 4},
        "Corridor": {"min_height": 2.3, "min_area": 1.5},
    }

    def _extract_quantities(space):
        vals = {}
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if not pdef:
                continue
            if pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    qname = (getattr(q, "Name", "") or "").lower()
                    if hasattr(q, "AreaValue"):
                        vals[qname] = float(q.AreaValue)
                    elif hasattr(q, "LengthValue"):
                        vals[qname] = float(q.LengthValue)
                    elif hasattr(q, "VolumeValue"):
                        vals[qname] = float(q.VolumeValue)
            elif pdef.is_a("IfcPropertySet"):
                for prop in getattr(pdef, "HasProperties", []) or []:
                    pname = (getattr(prop, "Name", "") or "").lower()
                    val = getattr(prop, "NominalValue", None)
                    wrapped = getattr(val, "wrappedValue", None)
                    if isinstance(wrapped, (int, float)):
                        vals[pname] = float(wrapped)
        return vals

    def _classify_space(name):
        n = (name or "").lower()
        if any(k in n for k in ["living", "salon", "sala", "menjador"]):
            return "Living Room"
        if any(k in n for k in ["bed", "bedroom", "habit", "dorm"]):
            return "Bedroom"
        if any(k in n for k in ["kitchen", "cuina", "cocina"]):
            return "Kitchen"
        if any(k in n for k in ["bath", "toilet", "lavabo", "wc", "bany"]):
            return "Bathroom"
        if any(k in n for k in ["corridor", "hall", "pasillo", "entry", "vestib", "distrib"]):
            return "Corridor"
        return None

    report = {"passed": [], "failed": [], "warnings": [], "summary": {}}

    for space in spaces:
        name = getattr(space, "Name", None) or "Unnamed"
        space_type = _classify_space(name)
        q = _extract_quantities(space)

        area = next((q[k] for k in ["netfloorarea", "grossfloorarea", "area", "net area"] if k in q), None)
        height = next((q[k] for k in ["height", "netheight", "clearheight", "unbounded height"] if k in q), None)
        if height is None and area and "grossvolume" in q and area > 0:
            height = q["grossvolume"] / area

        item = {
            "space": name,
            "space_type": space_type or "Unknown",
            "area": area,
            "height": height,
            "guid": getattr(space, "GlobalId", None),
        }

        reasons = []
        if not space_type:
            reasons.append("Space type could not be inferred from name")
        else:
            req = requirements[space_type]
            if area is None:
                reasons.append("Missing floor area")
            elif area < req["min_area"]:
                reasons.append(f"Area {area:.2f}m2 < required {req['min_area']}m2")

            if height is None:
                reasons.append("Missing room height")
            elif height < req["min_height"]:
                reasons.append(f"Height {height:.2f}m < required {req['min_height']}m")

        if reasons:
            item["reasons"] = reasons
            report["failed"].append(item)
            if not space_type:
                report["warnings"].append(f"{name}: unknown space type")
        else:
            report["passed"].append(item)

    total = len(spaces)
    passed = len(report["passed"])
    failed = len(report["failed"])
    report["summary"] = {
        "total_spaces": total,
        "passed_count": passed,
        "failed_count": failed,
        "compliance_rate": (passed / total) if total else 0.0,
        "overall_compliant": failed == 0,
    }
    return report


# Run and show output
result = check_space_compliance(spaces)
print("Exercise 1 - Space Compliance")
print(f"Total spaces: {result['summary']['total_spaces']}")
print(f"Passed: {result['summary']['passed_count']}")
print(f"Failed: {result['summary']['failed_count']}")
print(f"Compliance rate: {result['summary']['compliance_rate']:.1%}")
if result["failed"]:
    print("Sample failures:")
    for item in result["failed"][:5]:
        print(f"- {item['space']}: {', '.join(item['reasons'])}")


Passed: 0 spaces
Failed: 0 spaces


## Exercise 2: Window Detection and Compliance Verification

**Objective:** Find all windows in the model and verify they meet natural light and ventilation requirements.

**Reference:** [Catalan Building Code](https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e)

### Key Requirements

Residential spaces must have:
- **Minimum window area** = 1/8 of floor area (12.5%)
- **Minimum window dimensions** = 60cm width, 100cm height (for single opening)
- Living areas should have direct natural light

### Your Task

Write a function `analyze_window_compliance(ifc_model, spaces)` that:

1. **Finds all IfcWindow** entities in the model
2. **Links windows to spaces** (which room is each window in?)
3. **Extracts properties**: dimensions, area, orientation
4. **Calculates window-to-floor ratio** for each space
5. **Reports compliance** for each space with windows

**Hints:**
- Windows are `IfcWindow` entities
- To link windows to spaces, check spatial containment relationships
- Window area can be calculated from the frame/pane dimensions
- You may need to examine the geometric representation

### Starter Code

In [ ]:
# Exercise 2: Implement your solution here

def analyze_window_compliance(ifc_model, spaces):
    """
    Analyze windows in each space and check compliance.

    Args:
        ifc_model: The loaded IFC model
        spaces: List of IfcSpace objects

    Returns:
        dict: Report with window analysis and compliance status
    """

    def _extract_space_area(space):
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    n = (getattr(q, "Name", "") or "").lower()
                    if "area" in n and hasattr(q, "AreaValue"):
                        return float(q.AreaValue)
        return None

    def _window_dimensions(window):
        w = getattr(window, "OverallWidth", None)
        h = getattr(window, "OverallHeight", None)
        if w is not None and h is not None:
            return float(w), float(h)
        return None, None

    def _window_orientation(window):
        placement = getattr(window, "ObjectPlacement", None)
        rel = getattr(placement, "RelativePlacement", None) if placement else None
        axis = getattr(rel, "RefDirection", None) if rel else None
        ratios = getattr(axis, "DirectionRatios", None) if axis else None
        if ratios and len(ratios) >= 2:
            x = float(ratios[0])
            y = float(ratios[1])
            return "E/W" if abs(x) >= abs(y) else "N/S"
        return "Unknown"

    def _is_habitable(name):
        n = (name or "").lower()
        return any(k in n for k in ["living", "bed", "habit", "dorm", "kitchen", "salon", "sala"])

    windows = ifc_model.by_type("IfcWindow")

    report = {
        "total_windows": len(windows),
        "windows_by_space": {},
        "compliance_status": {},
        "unassigned_windows": []
    }

    for space in spaces:
        sname = getattr(space, "Name", None) or getattr(space, "GlobalId", "Unnamed")
        report["windows_by_space"][sname] = {"space_area": _extract_space_area(space), "windows": []}

    def _candidate_space_names(window):
        names = []
        for rel in getattr(window, "ContainedInStructure", []) or []:
            container = getattr(rel, "RelatingStructure", None)
            if container and container.is_a("IfcSpace"):
                names.append(getattr(container, "Name", None) or getattr(container, "GlobalId", None))
        for b in getattr(window, "ProvidesBoundaries", []) or []:
            sp = getattr(b, "RelatingSpace", None)
            if sp and sp.is_a("IfcSpace"):
                names.append(getattr(sp, "Name", None) or getattr(sp, "GlobalId", None))
        return [n for n in names if n]

    for window in windows:
        width, height = _window_dimensions(window)
        area = (width * height) if (width and height) else None
        window_data = {
            "name": getattr(window, "Name", None) or getattr(window, "GlobalId", "UnnamedWindow"),
            "width": width,
            "height": height,
            "area": area,
            "orientation": _window_orientation(window),
            "min_dimensions_ok": bool(width and height and width >= 0.6 and height >= 1.0),
        }

        linked = False
        for sname in _candidate_space_names(window):
            if sname in report["windows_by_space"]:
                report["windows_by_space"][sname]["windows"].append(window_data)
                linked = True
        if not linked:
            report["unassigned_windows"].append(window_data)

    for space in spaces:
        sname = getattr(space, "Name", None) or getattr(space, "GlobalId", "Unnamed")
        space_entry = report["windows_by_space"][sname]
        space_area = space_entry["space_area"]
        total_window_area = sum(w["area"] or 0 for w in space_entry["windows"])
        ratio = (total_window_area / space_area) if (space_area and space_area > 0) else None
        requires_window = _is_habitable(sname)

        compliant = True
        reasons = []

        if requires_window and not space_entry["windows"]:
            compliant = False
            reasons.append("No windows found in habitable space")

        if requires_window and ratio is not None and ratio < 0.125:
            compliant = False
            reasons.append(f"Window/floor ratio {ratio:.3f} < 0.125")

        if any(not w["min_dimensions_ok"] for w in space_entry["windows"]):
            compliant = False
            reasons.append("At least one window is below 0.6m x 1.0m")

        report["compliance_status"][sname] = {
            "space_area": space_area,
            "window_count": len(space_entry["windows"]),
            "total_window_area": total_window_area,
            "window_to_floor_ratio": ratio,
            "requires_window": requires_window,
            "compliant": compliant,
            "reasons": reasons,
        }

    return report


# Run and show output
analysis = analyze_window_compliance(ifc, spaces)
print("Exercise 2 - Window Compliance")
print(f"Total windows found: {analysis['total_windows']}")
print(f"Unassigned windows: {len(analysis['unassigned_windows'])}")
space_results = analysis["compliance_status"]
compliant_spaces = [k for k, v in space_results.items() if v["compliant"]]
non_compliant_spaces = [k for k, v in space_results.items() if not v["compliant"]]
print(f"Compliant spaces: {len(compliant_spaces)}")
print(f"Non-compliant spaces: {len(non_compliant_spaces)}")
for name in non_compliant_spaces[:5]:
    print(f"- {name}: {', '.join(space_results[name]['reasons'])}")


Found 24 windows in the model


## Bonus Exercise: Fire Safety Route Analysis

**Objective:** Find the longest evacuation route within the apartment and verify it meets fire safety requirements.

**Difficulty:** Advanced

**Reference:** [Catalan Building Code - Fire Safety Section](https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e)

### Fire Safety Requirements

According to the Catalan building code:
- **Maximum travel distance** to exit: ≤ 25-30 m (depending on building type)
- **Minimum corridor width**: 1.2 m
- **Minimum door width**: 0.8 m (for exits)
- **No dead-end corridors** longer than 10 m

### Your Task

Write a function `analyze_evacuation_routes(ifc_model, spaces)` that:

1. **Builds a spatial graph** of the apartment (rooms as nodes, doors/openings as connections)
2. **Calculates distances** between spaces (using area/perimeter as proxy)
3. **Finds the longest route** from any point to the nearest exit
4. **Validates** the route against fire safety requirements
5. **Identifies bottlenecks** (narrow corridors, small doors)

**Hints:**
- Think of spaces as nodes and doors as edges in a graph
- Use BFS/DFS to find longest paths
- Door dimensions can indicate width constraints
- Consider calculating distances based on space geometry
- This is a simplified model - real analysis would use detailed geometry

### Starter Code

```python
def analyze_evacuation_routes(ifc_model, spaces):
    """
    Analyze evacuation routes and fire safety compliance.
    
    Args:
        ifc_model: The loaded IFC model
        spaces: List of IfcSpace objects
        
    Returns:
        dict: Fire safety analysis and recommendations
    """
    
    # Get all doors (potential connections between spaces)
    doors = ifc_model.by_type("IfcDoor")
    
    analysis = {
        "total_spaces": len(spaces),
        "total_doors": len(doors),
        "longest_route": None,
        "longest_distance": 0,
        "safety_issues": [],
        "compliant": False
    }
    
    print(f"Analyzing {len(spaces)} spaces with {len(doors)} doors")
    
    # TODO: Build spatial connectivity graph
    # TODO: Find longest evacuation path
    # TODO: Check corridor widths and door dimensions
    # TODO: Validate against requirements
    # TODO: Report issues and recommendations
    
    return analysis


# Run the analysis (uncomment when ready)
# fire_analysis = analyze_evacuation_routes(ifc, spaces)
```

In [ ]:
# Bonus Exercise: Implement your solution here

from collections import deque, defaultdict
import math

def analyze_evacuation_routes(ifc_model, spaces):
    """
    Analyze evacuation routes and fire safety compliance.

    Args:
        ifc_model: The loaded IFC model
        spaces: List of IfcSpace objects

    Returns:
        dict: Fire safety analysis and recommendations
    """

    doors = ifc_model.by_type("IfcDoor")

    analysis = {
        "total_spaces": len(spaces),
        "total_doors": len(doors),
        "graph": {},
        "longest_route": None,
        "longest_distance": 0,
        "safety_issues": [],
        "compliant": False
    }

    def _space_name(space):
        return getattr(space, "Name", None) or getattr(space, "GlobalId", "Unnamed")

    names = [_space_name(s) for s in spaces]
    graph = defaultdict(set)

    def _door_connected_spaces(door):
        found = []
        for rel in getattr(door, "ProvidesBoundaries", []) or []:
            sp = getattr(rel, "RelatingSpace", None)
            if sp and sp.is_a("IfcSpace"):
                found.append(_space_name(sp))
        unique = []
        for n in found:
            if n not in unique:
                unique.append(n)
        return unique

    for door in doors:
        connected = _door_connected_spaces(door)
        if len(connected) >= 2:
            for i in range(len(connected)):
                for j in range(i + 1, len(connected)):
                    a, b = connected[i], connected[j]
                    graph[a].add(b)
                    graph[b].add(a)

    for i in range(len(names) - 1):
        if names[i] not in graph and names[i + 1] not in graph:
            graph[names[i]].add(names[i + 1])
            graph[names[i + 1]].add(names[i])

    exits = [n for n in names if any(k in n.lower() for k in ["entry", "entrance", "exit", "hall"])]
    if not exits and names:
        exits = [names[0]]

    assumed_edge_distance = 5.0

    def _min_distance_to_exit(start):
        seen = {start}
        queue = deque([(start, 0, [start])])
        while queue:
            node, depth, path = queue.popleft()
            if node in exits:
                return depth * assumed_edge_distance, path
            for nxt in graph[node]:
                if nxt not in seen:
                    seen.add(nxt)
                    queue.append((nxt, depth + 1, path + [nxt]))
        return math.inf, []

    for n in names:
        dist, path = _min_distance_to_exit(n)
        if dist != math.inf and dist > analysis["longest_distance"]:
            analysis["longest_distance"] = dist
            analysis["longest_route"] = path

    if analysis["longest_distance"] > 25:
        analysis["safety_issues"].append(f"Longest travel distance is {analysis['longest_distance']:.1f}m (> 25m)")

    for door in doors:
        width = getattr(door, "OverallWidth", None)
        if width is not None and float(width) < 0.8:
            dname = getattr(door, "Name", None) or getattr(door, "GlobalId", "UnnamedDoor")
            analysis["safety_issues"].append(f"Door '{dname}' width {float(width):.2f}m < 0.80m")

    for s in spaces:
        sname = _space_name(s)
        if "corridor" in sname.lower() or "hall" in sname.lower():
            width = None
            for rel in getattr(s, "IsDefinedBy", []) or []:
                pdef = getattr(rel, "RelatingPropertyDefinition", None)
                if pdef and pdef.is_a("IfcElementQuantity"):
                    for q in getattr(pdef, "Quantities", []) or []:
                        qname = (getattr(q, "Name", "") or "").lower()
                        if "width" in qname and hasattr(q, "LengthValue"):
                            width = float(q.LengthValue)
                            break
                if width is not None:
                    break
            if width is not None and width < 1.2:
                analysis["safety_issues"].append(f"Corridor '{sname}' width {width:.2f}m < 1.20m")

    for node, neighbors in graph.items():
        if node not in exits and len(neighbors) == 1:
            analysis["safety_issues"].append(f"Potential dead-end near '{node}'")

    analysis["graph"] = {k: sorted(v) for k, v in graph.items()}
    analysis["compliant"] = len(analysis["safety_issues"]) == 0
    return analysis


# Run and show output (bonus)
fire_analysis = analyze_evacuation_routes(ifc, spaces)
print("Bonus - Fire Safety Route Analysis")
print(f"Total spaces: {fire_analysis['total_spaces']}")
print(f"Total doors: {fire_analysis['total_doors']}")
print(f"Longest distance: {fire_analysis['longest_distance']:.1f} m")
print(f"Compliant: {fire_analysis['compliant']}")
if fire_analysis["safety_issues"]:
    print("Safety issues:")
    for issue in fire_analysis["safety_issues"][:8]:
        print(f"- {issue}")


## Useful IFC Concepts

### Common Entity Types

- **IfcSpace**: A room, area, or zone in the building
- **IfcWindow**: Windows for light and ventilation  
- **IfcDoor**: Doors, openings, access points
- **IfcWall**: Boundary elements
- **IfcBuildingElement**: General building components
- **IfcElement**: Physical building pieces with properties

### Accessing Properties

```python
# Get all entities of a type
elements = ifc.by_type("IfcWindow")

# Access properties
entity.Name                    # String name
entity.Description             # Text description
entity.ObjectPlacement        # Location/coordinates
entity.Representation         # Geometry data
entity.QuantityInSpace         # Calculate-derived quantities

# Common relationships
entity.HasProperties           # Get properties
entity.BoundedBy               # Spatial boundaries
entity.HostedBy                # Connection relationships
```

### Helpful Methods

```python
# Query by GUID
entity = ifc.by_id(guid)

# Filter by property value
elements = ifc.by_attribute("Name", "Kitchen")

# Get all instances of a type
spaces = ifc.by_type("IfcSpace")

# Check entity type
if space.is_a("IfcSpace"):
    print("This is a space")
```

## Resources for Help

- **IFC Knowledge Base**: https://notebooklm.google.com/notebook/0925c2a1-519b-40a8-aca4-1e832d219f3c
- **IfcOpenShell Documentation**: https://docs.ifcopenshell.org/ifcopenshell-python.html
- **BuildingSmart Standards**: https://www.buildingsmart.org/
- **Catalan Building Code**: https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e

## Next Steps

After completing these exercises, you'll have learned:
- ✓ How to load and explore IFC files
- ✓ How to extract spatial and building data
- ✓ How to validate designs against building codes
- ✓ How to work with doors, windows, and routes

These skills apply to real-world AEC (Architecture, Engineering, Construction) workflows.